In [16]:
!pip uninstall -y torchvision -q
!pip install -q -U transformers accelerate

In [1]:
import torch
import transformers

print("PyTorch sürümü:", torch.__version__)
print("Transformers sürümü:", transformers.__version__)
print("GPU kullanılabilir mi?:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch sürümü: 2.14.1+cu130
Transformers sürümü: 5.19.0
GPU kullanılabilir mi?: True
GPU: NVIDIA A100-SXM4-80GB


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Cihaz:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)
model.eval()

print("Model başarıyla yüklendi.")

Cihaz: cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model başarıyla yüklendi.


In [3]:
text = "Yapay zeka gelecekte"

inputs = tokenizer(
    text,
    return_tensors="pt"
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

print("Metin:")
print(text)

print("\nToken ID'leri:")
print(inputs["input_ids"])

print("\nTokenlar:")
print(
    tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0]
    )
)

Metin:
Yapay zeka gelecekte

Token ID'leri:
tensor([[   56,   391,   352,  1147, 52313, 97502,   346, 40951]],
       device='cuda:0')

Tokenlar:
['Y', 'ap', 'ay', 'Ġz', 'eka', 'Ġgele', 'ce', 'kte']


In [4]:
with torch.no_grad():
    outputs = model(**inputs)

print("Logits boyutu:")
print(outputs.logits.shape)

Logits boyutu:
torch.Size([1, 8, 151936])


In [5]:
next_token_logits = outputs.logits[0, -1, :]

probabilities = torch.softmax(
    next_token_logits,
    dim=-1
)

top_probs, top_ids = torch.topk(
    probabilities,
    k=10
)

print("Girdi:", text)
print("-" * 60)

for rank, (token_id, probability) in enumerate(
    zip(top_ids, top_probs),
    start=1
):

    token = tokenizer.decode(
        [token_id.item()]
    )

    print(
        f"{rank:2}. "
        f"{repr(token):20} "
        f"{probability.item()*100:8.3f}%"
    )

Girdi: Yapay zeka gelecekte
------------------------------------------------------------
 1. 'ki'                    7.086%
 2. ','                     4.575%
 3. ' bir'                  3.561%
 4. ' y'                    2.110%
 5. ' k'                    1.778%
 6. ' o'                    1.544%
 7. ' on'                   1.193%
 8. ' '                     0.997%
 9. ' de'                   0.846%
10. ' g'                    0.776%


In [6]:
def predict_next_tokens(text, top_k=10):

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    next_token_logits = outputs.logits[0, -1, :]

    probabilities = torch.softmax(
        next_token_logits,
        dim=-1
    )

    top_probs, top_ids = torch.topk(
        probabilities,
        k=top_k
    )

    print("=" * 70)
    print("INPUT:")
    print(text)

    print("\nNEXT TOKEN CANDIDATES")
    print("-" * 70)

    for rank, (token_id, probability) in enumerate(
        zip(top_ids, top_probs),
        start=1
    ):

        token = tokenizer.decode(
            [token_id.item()]
        )

        print(
            f"{rank:2}. "
            f"{repr(token):20} "
            f"{probability.item()*100:8.3f}%"
        )

In [7]:
predict_next_tokens(
    "Elektrik motorunda aşırı sıcaklık tespit edildi ve"
)

INPUT:
Elektrik motorunda aşırı sıcaklık tespit edildi ve

NEXT TOKEN CANDIDATES
----------------------------------------------------------------------
 1. ' bu'                   9.058%
 2. ' k'                    2.806%
 3. ' elek'                 2.118%
 4. ' y'                    1.275%
 5. ' h'                    1.207%
 6. ' b'                    1.152%
 7. ' bir'                  1.143%
 8. ' on'                   1.125%
 9. ' bun'                  1.049%
10. ' kend'                 0.948%


In [8]:
predict_next_tokens(
    "Bir rulman arızasının en yaygın nedeni"
)

INPUT:
Bir rulman arızasının en yaygın nedeni

NEXT TOKEN CANDIDATES
----------------------------------------------------------------------
 1. ','                     8.936%
 2. ' olan'                 8.527%
 3. 'yle'                   7.123%
 4. ' olduğunu'             5.637%
 5. ' ned'                  4.425%
 6. ' bu'                   2.835%
 7. ' olduğu'               2.602%
 8. ' olarak'               2.405%
 9. ' ol'                   2.368%
10. ' var'                  1.059%


In [9]:
predict_next_tokens(
    "Yapay zeka gelecekte"
)

INPUT:
Yapay zeka gelecekte

NEXT TOKEN CANDIDATES
----------------------------------------------------------------------
 1. 'ki'                    7.086%
 2. ','                     4.575%
 3. ' bir'                  3.561%
 4. ' y'                    2.110%
 5. ' k'                    1.778%
 6. ' o'                    1.544%
 7. ' on'                   1.193%
 8. ' '                     0.997%
 9. ' de'                   0.846%
10. ' g'                    0.776%
